In [1]:
import cv2
import numpy as np
import mediapipe as mp
import urllib.request
import os
import time
import math
from collections import deque

# ============================================================
# TRINETRA - ELDERLY SAFETY MONITOR
# Real-time:
# STANDING | SITTING | WALKING | LYING | POSSIBLE FALL
# ============================================================

print("Starting TRINETRA...")

# ------------------------------------------------------------
# 1. Download MediaPipe Pose model automatically
# ------------------------------------------------------------

MODEL_PATH = "pose_landmarker_lite.task"

MODEL_URL = (
    "https://storage.googleapis.com/"
    "mediapipe-models/pose_landmarker/"
    "pose_landmarker_lite/float16/latest/"
    "pose_landmarker_lite.task"
)

if not os.path.exists(MODEL_PATH):
    print("Downloading AI pose model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Model downloaded.")

# ------------------------------------------------------------
# 2. Create MediaPipe Pose Landmarker
# ------------------------------------------------------------

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=MODEL_PATH
    ),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

landmarker = PoseLandmarker.create_from_options(options)

print("MediaPipe Pose model loaded successfully.")

# ------------------------------------------------------------
# 3. Camera
# ------------------------------------------------------------

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("❌ Camera could not be opened.")
    print("Check Windows camera permission or close other camera apps.")
    landmarker.close()
    raise RuntimeError("Webcam unavailable")

print("✅ Camera connected.")
print("Press Q to stop.")

# ------------------------------------------------------------
# 4. Temporal history
# ------------------------------------------------------------

activity_history = deque(maxlen=15)
center_history = deque(maxlen=15)
time_history = deque(maxlen=15)

previous_activity = "ANALYZING"
fall_detected = False
fall_time = None

# ------------------------------------------------------------
# 5. Helper functions
# ------------------------------------------------------------

def midpoint(a, b):
    return (
        (a[0] + b[0]) / 2,
        (a[1] + b[1]) / 2
    )


def point_distance(a, b):
    return math.sqrt(
        (a[0] - b[0]) ** 2 +
        (a[1] - b[1]) ** 2
    )


def calculate_angle(a, b, c):

    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    ba = a - b
    bc = c - b

    denominator = (
        np.linalg.norm(ba) *
        np.linalg.norm(bc)
    )

    if denominator == 0:
        return 0

    cosine = np.dot(ba, bc) / denominator

    cosine = np.clip(
        cosine,
        -1.0,
        1.0
    )

    return np.degrees(
        np.arccos(cosine)
    )


# ------------------------------------------------------------
# 6. Extract important landmarks
# ------------------------------------------------------------

def get_landmarks(landmarks):

    # MediaPipe landmark indexes
    LEFT_SHOULDER = 11
    RIGHT_SHOULDER = 12

    LEFT_HIP = 23
    RIGHT_HIP = 24

    LEFT_KNEE = 25
    RIGHT_KNEE = 26

    LEFT_ANKLE = 27
    RIGHT_ANKLE = 28

    ls = landmarks[LEFT_SHOULDER]
    rs = landmarks[RIGHT_SHOULDER]

    lh = landmarks[LEFT_HIP]
    rh = landmarks[RIGHT_HIP]

    lk = landmarks[LEFT_KNEE]
    rk = landmarks[RIGHT_KNEE]

    la = landmarks[LEFT_ANKLE]
    ra = landmarks[RIGHT_ANKLE]

    shoulder = midpoint(
        (ls.x, ls.y),
        (rs.x, rs.y)
    )

    hip = midpoint(
        (lh.x, lh.y),
        (rh.x, rh.y)
    )

    knee = midpoint(
        (lk.x, lk.y),
        (rk.x, rk.y)
    )

    ankle = midpoint(
        (la.x, la.y),
        (ra.x, ra.y)
    )

    return (
        shoulder,
        hip,
        knee,
        ankle
    )


# ------------------------------------------------------------
# 7. Posture classifier
# ------------------------------------------------------------

def classify_posture(landmarks):

    shoulder, hip, knee, ankle = get_landmarks(
        landmarks
    )

    # Body dimensions
    vertical_height = abs(
        shoulder[1] - ankle[1]
    )

    horizontal_width = abs(
        ankle[0] - shoulder[0]
    )

    # Torso orientation
    dx = hip[0] - shoulder[0]
    dy = hip[1] - shoulder[1]

    torso_angle = abs(
        math.degrees(
            math.atan2(dx, dy)
        )
    )

    # Knee angles
    left_knee_angle = calculate_angle(
        (landmarks[11].x, landmarks[11].y),
        (landmarks[25].x, landmarks[25].y),
        (landmarks[27].x, landmarks[27].y)
    )

    right_knee_angle = calculate_angle(
        (landmarks[12].x, landmarks[12].y),
        (landmarks[26].x, landmarks[26].y),
        (landmarks[28].x, landmarks[28].y)
    )

    knee_angle = (
        left_knee_angle +
        right_knee_angle
    ) / 2

    # --------------------------------------------------------
    # LYING
    # --------------------------------------------------------

    if (
        horizontal_width >
        vertical_height * 1.25
        and torso_angle > 45
    ):
        return "LYING"

    # --------------------------------------------------------
    # SITTING
    # --------------------------------------------------------

    if knee_angle < 135:
        return "SITTING"

    # --------------------------------------------------------
    # STANDING
    # --------------------------------------------------------

    if (
        knee_angle >= 135
        and torso_angle < 35
    ):
        return "STANDING"

    return "ANALYZING"


# ------------------------------------------------------------
# 8. Movement calculation
# ------------------------------------------------------------

def calculate_movement():

    if len(center_history) < 5:
        return 0

    old_point = center_history[-5]
    new_point = center_history[-1]

    old_time = time_history[-5]
    new_time = time_history[-1]

    elapsed = new_time - old_time

    if elapsed <= 0:
        return 0

    movement = point_distance(
        old_point,
        new_point
    )

    return movement / elapsed


# ------------------------------------------------------------
# 9. Fall detection
# ------------------------------------------------------------

def detect_fall():

    if len(activity_history) < 10:
        return False

    recent = list(activity_history)

    # Current posture
    current = recent[-1]

    if current != "LYING":
        return False

    # Was the person previously upright?
    previous_upright = any(
        x in ["STANDING", "WALKING"]
        for x in recent[:-5]
    )

    if not previous_upright:
        return False

    # Person must remain lying for several frames
    lying_frames = recent[-5:].count("LYING")

    if lying_frames < 3:
        return False

    # Look for body-center displacement
    if len(center_history) < 8:
        return False

    old_y = center_history[-8][1]
    new_y = center_history[-3][1]

    vertical_change = abs(
        new_y - old_y
    )

    # Strong downward movement
    rapid_downward_movement = (
        vertical_change > 0.07
    )

    return rapid_downward_movement


# ------------------------------------------------------------
# 10. Draw pose landmarks
# ------------------------------------------------------------

def draw_pose(frame, landmarks):

    h, w, _ = frame.shape

    connections = [
        (11, 12),

        (11, 13),
        (13, 15),

        (12, 14),
        (14, 16),

        (11, 23),
        (12, 24),

        (23, 24),

        (23, 25),
        (25, 27),

        (24, 26),
        (26, 28)
    ]

    # Draw points
    for landmark in landmarks:

        x = int(landmark.x * w)
        y = int(landmark.y * h)

        if (
            0 <= x < w
            and 0 <= y < h
        ):
            cv2.circle(
                frame,
                (x, y),
                5,
                (0, 255, 0),
                -1
            )

    # Draw skeleton
    for start, end in connections:

        x1 = int(
            landmarks[start].x * w
        )

        y1 = int(
            landmarks[start].y * h
        )

        x2 = int(
            landmarks[end].x * w
        )

        y2 = int(
            landmarks[end].y * h
        )

        cv2.line(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )


# ============================================================
# 11. MAIN CAMERA LOOP
# ============================================================

timestamp_ms = 0

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        print("Camera frame unavailable.")
        break

    # Mirror webcam
    frame = cv2.flip(frame, 1)

    # Convert OpenCV BGR → RGB
    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    # MediaPipe image
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    timestamp_ms += 33

    # Pose detection
    result = landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )

    activity = "NO PERSON DETECTED"
    confidence = 0

    # --------------------------------------------------------
    # PERSON DETECTED
    # --------------------------------------------------------

    if result.pose_landmarks:

        landmarks = result.pose_landmarks[0]

        # Average landmark visibility
        visibility_values = []

        for landmark in landmarks:

            if hasattr(
                landmark,
                "visibility"
            ):
                visibility_values.append(
                    landmark.visibility
                )

        if visibility_values:

            confidence = int(
                np.mean(
                    visibility_values
                ) * 100
            )

        # Posture
        posture = classify_posture(
            landmarks
        )

        # Body center
        shoulder, hip, _, _ = get_landmarks(
            landmarks
        )

        body_center = midpoint(
            shoulder,
            hip
        )

        center_history.append(
            body_center
        )

        time_history.append(
            time.time()
        )

        # Movement
        movement = calculate_movement()

        # Walking
        if (
            posture == "STANDING"
            and movement > 0.035
        ):
            activity = "WALKING"

        else:
            activity = posture

        # Store activity
        activity_history.append(
            activity
        )

        # Fall detection
        if detect_fall():

            fall_detected = True

            if fall_time is None:
                fall_time = time.time()

        # Keep fall alert for several seconds
        if fall_detected:

            if (
                fall_time is not None
                and time.time() - fall_time > 8
            ):

                fall_detected = False
                fall_time = None

        # Draw skeleton
        draw_pose(
            frame,
            landmarks
        )

    else:

        activity = "NO PERSON DETECTED"

        confidence = 0

        activity_history.clear()
        center_history.clear()
        time_history.clear()

        fall_detected = False
        fall_time = None

    # ========================================================
    # TRINETRA DISPLAY
    # ========================================================

    # Top information panel
    cv2.rectangle(
        frame,
        (15, 15),
        (470, 190),
        (25, 25, 25),
        -1
    )

    cv2.putText(
        frame,
        "TRINETRA",
        (35, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        "ELDERLY SAFETY MONITOR",
        (35, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (200, 200, 200),
        2
    )

    cv2.putText(
        frame,
        f"Activity: {activity}",
        (35, 115),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Confidence: {confidence}%",
        (35, 145),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        "Camera: CONNECTED",
        (35, 175),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (100, 255, 100),
        2
    )

    # ========================================================
    # FALL ALERT
    # ========================================================

    if fall_detected:

        # Alert box
        cv2.rectangle(
            frame,
            (0, 420),
            (frame.shape[1], frame.shape[0]),
            (0, 0, 160),
            -1
        )

        cv2.putText(
            frame,
            "!!! POSSIBLE FALL !!!",
            (45, 475),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.1,
            (255, 255, 255),
            3
        )

        cv2.putText(
            frame,
            "VERIFICATION REQUIRED",
            (45, 515),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            "TRINETRA ALERT",
            (45, 550),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2
        )

    # --------------------------------------------------------
    # Displayq
    # --------------------------------------------------------

    cv2.imshow(
        "TRINETRA - Elderly Safety Monitor",
        frame
    )

    # Q = Quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# ============================================================
# CLEANUP
# ============================================================

cap.release()
cv2.destroyAllWindows()
landmarker.close()

print("TRINETRA monitoring stopped.")

Starting TRINETRA...
MediaPipe Pose model loaded successfully.
✅ Camera connected.
Press Q to stop.
TRINETRA monitoring stopped.


In [2]:
import time
from datetime import datetime
import threading
import platform

# ============================================================
# TRINETRA - ELDERLY CARE ALERT & REMINDER SYSTEM
# ============================================================

print("=" * 65)
print("              👁 TRINETRA")
print("        AI-POWERED ELDERLY SAFETY SYSTEM")
print("=" * 65)


# ============================================================
# CAREGIVER INFORMATION
# ============================================================

caregiver_name = input("\nEnter family/caregiver name: ")
caregiver_phone = input("Enter caregiver phone number: ")

print("\n✅ Caregiver registered")
print("Name :", caregiver_name)
print("Phone:", caregiver_phone)


# ============================================================
# REMINDER STORAGE
# ============================================================

reminders = []


# ============================================================
# TIME CONVERSION
# Converts 08:00 PM → hour=20, minute=0
# ============================================================

def convert_time(time_string):

    try:
        time_obj = datetime.strptime(
            time_string.strip().upper(),
            "%I:%M %p"
        )

        return time_obj.hour, time_obj.minute

    except ValueError:

        print("\n❌ Invalid time format.")
        print("Please enter time like: 08:00 AM")
        return None, None


# ============================================================
# ADD MEDICATION
# ============================================================

def add_medication():

    print("\n" + "-" * 65)
    print("💊 ADD MEDICATION")
    print("-" * 65)

    medicine = input("Medicine name: ")

    dosage = input(
        "Dosage (example: 1 Tablet): "
    )

    time_string = input(
        "When should it be taken? (example: 08:00 AM): "
    )

    hour, minute = convert_time(time_string)

    if hour is None:
        return

    repeat = input(
        "Repeat? (Daily / Once): "
    ).strip().capitalize()

    if repeat not in ["Daily", "Once"]:
        repeat = "Daily"

    reminder = {

        "type": "MEDICATION",

        "description": medicine,

        "dosage": dosage,

        "hour": hour,

        "minute": minute,

        "display_time": datetime.strptime(
            time_string.strip().upper(),
            "%I:%M %p"
        ).strftime("%I:%M %p"),

        "repeat": repeat,

        "completed": False,

        "last_triggered": None
    }

    reminders.append(reminder)

    print("\n✅ Medication reminder added")

    print("Medicine :", medicine)
    print("Dosage   :", dosage)
    print("Time     :", reminder["display_time"])
    print("Repeat   :", repeat)


# ============================================================
# ADD FOOD REMINDER
# ============================================================

def add_food():

    print("\n" + "-" * 65)
    print("🍱 ADD MEAL REMINDER")
    print("-" * 65)

    meal_name = input(
        "Meal name (example: Lunch): "
    )

    time_string = input(
        "Meal time (example: 01:00 PM): "
    )

    hour, minute = convert_time(time_string)

    if hour is None:
        return

    reminder = {

        "type": "FOOD",

        "description": meal_name,

        "dosage": "",

        "hour": hour,

        "minute": minute,

        "display_time": datetime.strptime(
            time_string.strip().upper(),
            "%I:%M %p"
        ).strftime("%I:%M %p"),

        "repeat": "Daily",

        "completed": False,

        "last_triggered": None
    }

    reminders.append(reminder)

    print("\n✅ Meal reminder added")

    print("Meal :", meal_name)
    print("Time :", reminder["display_time"])


# ============================================================
# ADD WATER REMINDER
# ============================================================

def add_water():

    print("\n" + "-" * 65)
    print("💧 ADD WATER REMINDER")
    print("-" * 65)

    time_string = input(
        "Water reminder time (example: 03:30 PM): "
    )

    hour, minute = convert_time(time_string)

    if hour is None:
        return

    reminder = {

        "type": "WATER",

        "description": "Drink Water",

        "dosage": "",

        "hour": hour,

        "minute": minute,

        "display_time": datetime.strptime(
            time_string.strip().upper(),
            "%I:%M %p"
        ).strftime("%I:%M %p"),

        "repeat": "Daily",

        "completed": False,

        "last_triggered": None
    }

    reminders.append(reminder)

    print("\n✅ Water reminder added")

    print("Reminder : Drink Water")
    print("Time     :", reminder["display_time"])


# ============================================================
# NOTIFICATION
# ============================================================

def browser_style_notification(title, message):

    print("\n" + "!" * 65)

    print("🔔 TRINETRA NOTIFICATION")

    print("!" * 65)

    print(title)

    print(message)

    print("!" * 65)


# ============================================================
# FAMILY EMERGENCY ALERT
# ============================================================

def send_family_alert(reason):

    print("\n")
    print("🚨" * 20)

    print("        TRINETRA EMERGENCY ALERT")

    print("🚨" * 20)

    print("Reason :", reason)

    print("Family :", caregiver_name)

    print("Phone  :", caregiver_phone)

    print(
        "Time   :",
        datetime.now().strftime("%I:%M:%S %p")
    )

    
    print(
        "Local alert system is active."
    )


# ============================================================
# FALL ALERT
# ============================================================

def possible_fall_alert():

    browser_style_notification(

        "🚨 POSSIBLE FALL DETECTED",

        "TRINETRA detected a possible fall."
    )

    print("\n")

    print("Does the elderly person need help?")

    print()

    print("1 → I'M OK")

    print("2 → NEED HELP")

    choice = input(
        "\nEnter choice: "
    )

    if choice == "1":

        print("\n✅ User confirmed: I'M OK")

        print(
            "Emergency alert cancelled."
        )

    elif choice == "2":

        print("\n🚨 USER REQUESTED HELP")

        send_family_alert(

            "Possible fall - elderly person requested assistance"
        )

    else:

        print(
            "\n⚠️ No valid response."
        )

        print(
            "Continue monitoring."
        )


# ============================================================
# CHECK REMINDERS
# ============================================================

def check_reminders():

    now = datetime.now()

    for reminder in reminders:

        if (
            now.hour == reminder["hour"]
            and now.minute == reminder["minute"]
        ):

            current_key = now.strftime(
                "%Y-%m-%d %H:%M"
            )

            if reminder["last_triggered"] != current_key:

                reminder["last_triggered"] = current_key


                # ---------------- MEDICATION ----------------

                if reminder["type"] == "MEDICATION":

                    browser_style_notification(

                        "💊 MEDICATION REMINDER",

                        f"Time for {reminder['description']} "
                        f"({reminder['dosage']})"
                    )


                # ---------------- FOOD ----------------

                elif reminder["type"] == "FOOD":

                    browser_style_notification(

                        "🍱 MEAL REMINDER",

                        f"It is time for {reminder['description']}."
                    )


                # ---------------- WATER ----------------

                elif reminder["type"] == "WATER":

                    browser_style_notification(

                        "💧 HYDRATION REMINDER",

                        "It is time to drink water."
                    )


# ============================================================
# DISPLAY MEDICATIONS
# ============================================================

def display_medications():

    print("\n")
    print("💊 TODAY'S MEDICATIONS")

    print("=" * 65)

    medications = [

        r for r in reminders
        if r["type"] == "MEDICATION"
    ]

    if not medications:

        print("No medications added.")

        return

    # Sort by time

    medications.sort(
        key=lambda x: (x["hour"], x["minute"])
    )

    for medicine in medications:

        if medicine["completed"]:

            status = "🟢 Taken"

        else:

            status = "🟡 Upcoming"

        print("\n" + "┌" + "─" * 63 + "┐")

        print(
            f"│ {medicine['display_time']:<61}│"
        )

        print(
            f"│ 💊 {medicine['description']:<57}│"
        )

        print(
            f"│ {medicine['dosage']:<61}│"
        )

        print(
            f"│ {status:<61}│"
        )

        print(
            "└" + "─" * 63 + "┘"
        )


# ============================================================
# DISPLAY ALL REMINDERS
# ============================================================

def display_all_reminders():

    print("\n")
    print("=" * 65)

    print("              ACTIVE REMINDERS")

    print("=" * 65)

    for reminder in sorted(
        reminders,
        key=lambda x: (x["hour"], x["minute"])
    ):

        print(

            f"{reminder['type']:12} | "
            f"{reminder['description']:20} | "
            f"{reminder['display_time']:10}"
        )


# ============================================================
# MEDICATION SETUP
# ============================================================

print("\n")
print("=" * 65)

print("             MEDICATION SETUP")

print("=" * 65)

print("\nYou can add medications one by one.")

print("Example:")
print("Medicine : BP Tablet")
print("Dosage   : 1 Tablet")
print("Time     : 08:00 AM")


while True:

    add_medication()

    more = input(
        "\nAdd another medication? (yes/no): "
    ).strip().lower()

    if more != "yes":
        break


# ============================================================
# FOOD SETUP
# ============================================================

print("\n")
print("=" * 65)

print("               MEAL SETUP")

print("=" * 65)

add_food()


# ============================================================
# WATER SETUP
# ============================================================

print("\n")
print("=" * 65)

print("             HYDRATION SETUP")

print("=" * 65)

add_water()


# ============================================================
# SYSTEM READY
# ============================================================

print("\n")
print("=" * 65)

print("       👁 TRINETRA ELDERLY CARE SYSTEM READY")

print("=" * 65)


print("\n✓ Caregiver information")

print("✓ Medication reminders")

print("✓ Food reminders")

print("✓ Water reminders")

print("✓ Fall emergency alert")

print("✓ Local notification system")

print("✓ Family alert architecture")


# ============================================================
# DISPLAY MEDICATIONS
# ============================================================

display_medications()
print("\nTRINETRA monitoring system is ready.")

              👁 TRINETRA
        AI-POWERED ELDERLY SAFETY SYSTEM



Enter family/caregiver name:  Priya
Enter caregiver phone number:  9876543210



✅ Caregiver registered
Name : Priya
Phone: 9876543210


             MEDICATION SETUP

You can add medications one by one.
Example:
Medicine : BP Tablet
Dosage   : 1 Tablet
Time     : 08:00 AM

-----------------------------------------------------------------
💊 ADD MEDICATION
-----------------------------------------------------------------


Medicine name:  Diabetes
Dosage (example: 1 Tablet):  1
When should it be taken? (example: 08:00 AM):  8:00 AM
Repeat? (Daily / Once):  daily



✅ Medication reminder added
Medicine : Diabetes
Dosage   : 1
Time     : 08:00 AM
Repeat   : Daily


...



               MEAL SETUP

-----------------------------------------------------------------
🍱 ADD MEAL REMINDER
-----------------------------------------------------------------


Meal name (example: Lunch):  Lunch
Meal time (example: 01:00 PM):  1:00 PM



✅ Meal reminder added
Meal : Lunch
Time : 01:00 PM


             HYDRATION SETUP

-----------------------------------------------------------------
💧 ADD WATER REMINDER
-----------------------------------------------------------------


Water reminder time (example: 03:30 PM):  2:30 PM



✅ Water reminder added
Reminder : Drink Water
Time     : 02:30 PM


       👁 TRINETRA ELDERLY CARE SYSTEM READY

✓ Caregiver information
✓ Medication reminders
✓ Food reminders
✓ Water reminders
✓ Fall emergency alert
✓ Local notification system
✓ Family alert architecture


💊 TODAY'S MEDICATIONS

┌───────────────────────────────────────────────────────────────┐
│ 08:00 AM                                                     │
│ 💊 Diabetes                                                 │
│ 1                                                            │
│ 🟡 Upcoming                                                   │
└───────────────────────────────────────────────────────────────┘

┌───────────────────────────────────────────────────────────────┐
│ 10:00 PM                                                     │
│ 💊 Vitamin Tablet                                           │
│ 1                                                            │
│ 🟡 Upcoming                                      

In [3]:
!pip install pygame


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pygame
import time

pygame.mixer.init()

def trinetra_alarm():
    print("\n" + "=" * 60)
    print("🚨🚨🚨 TRINETRA EMERGENCY ALERT 🚨🚨🚨")
    print("=" * 60)
    print("⚠️ FALL DETECTED!")
    print("Elderly person may need immediate assistance.")
    print()
    print("🔊 ALARM ACTIVATED")
    print("👤 Caregiver: Ram")
    print("☎️ 9948281314")
    print("=" * 60)

    # Generate an alarm using the computer speaker
    frequency = 1000
    duration = 0.5

    import numpy as np
    from scipy.io.wavfile import write

    sample_rate = 44100
    t = np.linspace(0, duration, int(sample_rate * duration), False)

    tone = np.sin(2 * np.pi * frequency * t)
    tone = (tone * 32767).astype(np.int16)

    write("trinetra_alarm.wav", sample_rate, tone)

    pygame.mixer.music.load("trinetra_alarm.wav")
    pygame.mixer.music.play()

    time.sleep(2)
    pygame.mixer.music.stop()

    print("✅ Emergency alarm completed.")

pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [7]:
import numpy as np
import pygame
from scipy.io.wavfile import write
import os
import tempfile

def trinetra_alarm():
    sample_rate = 44100
    duration = 2
    frequency = 1000

    t = np.linspace(
        0,
        duration,
        int(sample_rate * duration),
        endpoint=False
    )

    tone = np.sin(2 * np.pi * frequency * t)
    tone = (tone * 32767).astype(np.int16)

    # Save in a temporary folder where Python has permission
    alarm_file = os.path.join(
        tempfile.gettempdir(),
        "trinetra_alarm.wav"
    )

    write(alarm_file, sample_rate, tone)

    pygame.mixer.init()
    pygame.mixer.music.load(alarm_file)
    pygame.mixer.music.play()

    print("🚨 TRINETRA ALARM PLAYING 🚨")

In [8]:
trinetra_alarm()

🚨 TRINETRA ALARM PLAYING 🚨
